# Phase 07b — Multimodal Retrieval

**Main question:** How can a textual query retrieve the original image directly?

In Phase 07 we learned to describe figures with a VLM and retrieve them via text similarity.
That approach depends on VLM availability and treats captions as a lossy proxy.

In this notebook we add a second, complementary path:
embed images *directly* using CLIP, then fuse text + visual rankings with RRF.

**Sections:**
1. CLIP intuition — shared embedding space
2. Image embeddings — shape, dtype, normalization
3. Text-to-image retrieval — queries against the visual index
4. Visual vector index — the production `VisualVectorStore`
5. Parallel multimodal retrieval — text and visual side by side
6. Reciprocal Rank Fusion — combining rankings without score comparison
7. Compare retrieval methods — text-only, caption, CLIP, fused

**Package:** `mrta-rag[retrieval,multimodal]`

In [ ]:
# Standard setup — run once
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython.display import display
from PIL import Image
import io

SAMPLE_PDF = Path("../../tests/fixtures/sample.pdf")
assert SAMPLE_PDF.exists(), f"Sample PDF not found at {SAMPLE_PDF}"
print(f"Sample PDF: {SAMPLE_PDF.resolve()}")

---
## 07b.1 CLIP Intuition

CLIP (Contrastive Language-Image Pre-training) trains an image encoder and a text encoder
jointly so that matching image–text pairs end up **close** in a shared embedding space.

```
"a transformer architecture diagram"  →  text encoder  →  [0.12, -0.34, ...] (512-dim)
                                                                   ↕  dot product
   <diagram image>                    →  image encoder →  [0.11, -0.36, ...] (512-dim)
```

Both vectors are L2-normalised, so **dot product = cosine similarity**.

Key properties:
- Same vector space for images and text — cross-modal similarity is well-defined.
- CLIP scores are NOT calibrated probabilities; they cannot be compared with
  sentence-transformer scores from a different model.
- Fusion across models must happen at the **rank level**, not by averaging raw scores.

In [ ]:
from mrta import CLIPEmbedder

clip = CLIPEmbedder()  # loads ViT-B-32/openai (~350 MB on first run)
print(f"Model  : {clip.model_name}")
print(f"Dim    : {clip.dim}")

---
## 07b.2 Image Embeddings

Let us extract figures from the sample paper and inspect their CLIP embeddings.

In [ ]:
from mrta import load_pdf, extract_figures

doc = load_pdf(SAMPLE_PDF)
figures = extract_figures(doc)
print(f"Extracted {len(figures)} figures")
for f in figures:
    print(f"  page={f.page}  figure_index={f.figure_index}  {f.width}x{f.height}px")

In [ ]:
from mrta import EvidenceRecord

visual_records = [f.to_evidence_record() for f in figures]

if visual_records:
    sample_rec = visual_records[0]
    vec = clip.embed_image(sample_rec.to_pil())
    print(f"Vector shape : {vec.shape}")
    print(f"dtype        : {vec.dtype}")
    print(f"L2 norm      : {np.linalg.norm(vec):.4f}  (should be ~1.0)")
    print(f"First 5 dims : {vec[:5]}")
else:
    print("No figures found in sample PDF — this is expected for vector-graphic papers.")
    print("The rest of this notebook will use synthetic examples.")

In [ ]:
# Embed all available figures
if visual_records:
    embeddings = []
    for rec in visual_records:
        v = clip.embed_image(rec.to_pil())
        embeddings.append(v)
        print(f"  figure_index={rec.figure_index}  page={rec.page}  norm={np.linalg.norm(v):.4f}")

    E = np.stack(embeddings)  # shape: (n_figures, 512)
    print(f"\nEmbedding matrix: {E.shape}")

---
## 07b.3 Text-to-Image Retrieval

CLIP text and image embeddings live in the same space.
Cosine similarity between a text query and image vectors gives us cross-modal retrieval
**without any VLM or caption**.

In [ ]:
def text_to_image_search(query: str, records, embeddings_matrix, k: int = 3):
    """Manual text→image search using CLIP embeddings."""
    q_vec = clip.embed_text(query)  # (512,)
    scores = embeddings_matrix @ q_vec  # dot product = cosine similarity (both L2-norm)
    top_k = np.argsort(scores)[::-1][:k]
    return [(records[i], float(scores[i])) for i in top_k]


def display_results(query: str, results):
    print(f"Query: '{query}'")
    print("-" * 50)
    for rank, (rec, score) in enumerate(results, 1):
        print(f"  [{rank}] page={rec.page}  figure={rec.figure_index}  score={score:.4f}")
    print()

In [ ]:
# Run queries only if figures were extracted
if visual_records and len(visual_records) > 0:
    queries = [
        "diagram showing encoder and decoder",
        "attention mechanism architecture",
        "graph comparing model performance",
    ]
    for q in queries:
        results = text_to_image_search(q, visual_records, E, k=min(3, len(visual_records)))
        display_results(q, results)
else:
    print("Skipping — no raster figures available. See tutorial notebook for a synthetic example.")

---
## 07b.4 Visual Vector Index

The manual search above is fine for exploration, but it scans all embeddings every time.
For production we use `VisualVectorStore`, which wraps a FAISS `IndexFlatIP` index.

**Why is this index separate from the text and caption stores?**

| Store | Embedder | Dim | Space |
|---|---|---|---|
| `VectorStore` | sentence-transformers | 384 | text semantic |
| `CaptionVectorStore` | sentence-transformers | 384 | text semantic |
| `VisualVectorStore` | CLIP | 512 | image-text CLIP |

> Sentence-transformer and CLIP cosine scores are **not on the same scale**.
> Directly comparing or averaging them is statistically invalid.
> Fusion must happen at the **rank level** — which is exactly what RRF does.

In [ ]:
from mrta import VisualVectorStore

visual_store = VisualVectorStore(clip)

if visual_records:
    visual_store.add(visual_records)
    print(f"Visual index size: {visual_store.size}")

    query = "encoder decoder architecture"
    results = visual_store.search(query, k=3)
    print(f"\nQuery: '{query}'")
    for r in results:
        print(f"  page={r.page}  figure={r.figure_index}  score={r.retrieval_score:.4f}")
else:
    print("Visual store is empty — no raster figures extracted.")

---
## 07b.5 Parallel Multimodal Retrieval

Now we run text retrieval and visual retrieval **independently** on the same query
and inspect both result sets before fusing them.

In [ ]:
from mrta import Embedder, VectorStore, chunk_pdf

# Build text index
embedder = Embedder()
chunks = chunk_pdf(doc)
text_store = VectorStore(embedder)
text_store.add(chunks)
print(f"Text index size : {text_store.size} chunks")

In [ ]:
query = "What is the role of the attention mechanism?"

# --- Text retrieval ---
text_results = text_store.search_with_scores(query, k=5)
print("Text results:")
for rank, (chunk, score) in enumerate(text_results, 1):
    print(f"  [{rank}] page={chunk.page}  score={score:.4f}  '{chunk.text[:60]}...'")

print()

# --- Visual retrieval ---
if visual_store.size > 0:
    visual_results = visual_store.search_with_scores(query, k=5)
    print("Visual results:")
    for rank, (rec, score) in enumerate(visual_results, 1):
        print(f"  [{rank}] page={rec.page}  figure={rec.figure_index}  score={score:.4f}")
else:
    print("Visual results: (empty — no raster figures)")

Note that text scores (~0.3–0.9) and CLIP scores (~0.15–0.35) are not on the same scale.
We cannot directly compare or merge them. This is where RRF comes in.

---
## 07b.6 Reciprocal Rank Fusion

RRF assigns each document a fused score based on **rank position**, not raw score:

$$\text{RRF}(d) = \sum_{r \in \text{lists}} \frac{1}{k + \text{rank}_r(d)}$$

- $k = 60$ by default — smooths the advantage of top-ranked items.
- Documents present in multiple lists accumulate higher scores.
- Documents absent from a list contribute 0 for that list.

### Hand-calculated example

Suppose we have:

| doc | text rank | visual rank |
|-----|-----------|-------------|
| A   | 1         | 3           |
| B   | 2         | —           |
| C   | —         | 1           |

With k=60:
- A: 1/61 + 1/63 ≈ 0.0164 + 0.0159 = **0.0323**
- B: 1/62 ≈ **0.0161**
- C: 1/61 ≈ **0.0164**

Final order: A (0.0323) > C (0.0164) > B (0.0161)

In [ ]:
# Reproduce the hand-calculated example
from mrta import EvidenceRecord, reciprocal_rank_fusion

def make_ev(eid, page=1):
    return EvidenceRecord(
        evidence_id=eid, doc_id="doc", source="ex.pdf", page=page, modality="text", text=eid
    )

doc_A = make_ev("A", page=2)
doc_B = make_ev("B", page=4)
doc_C = make_ev("C", page=7)

results = reciprocal_rank_fusion(
    named_lists={"text": [doc_A, doc_B], "visual": [doc_C, doc_A]},
    k=60,
)

print("Fused ranking:")
for r in results:
    trank = r.per_list_rank.get("text", "—")
    vrank = r.per_list_rank.get("visual", "—")
    print(f"  {r.record.evidence_id}  rrf={r.rrf_score:.4f}  text_rank={trank}  visual_rank={vrank}")

In [ ]:
# Now fuse real retrieval results using MultimodalRetriever
from mrta import MultimodalRetriever

retriever = MultimodalRetriever(
    vector_store=text_store,
    visual_store=visual_store if visual_store.size > 0 else None,
    rrf_k=60,
)

query = "What is the role of the attention mechanism?"
fused = retriever.retrieve_with_fusion_details(query, k_text=5, k_visual=5, k_final=8)

print(f"Query: '{query}'\n")
print(f"{'Rank':<6}{'ID':<30}{'Modality':<10}{'RRF':<10}{'Text':<8}{'Visual'}")
print("-" * 72)
for rank, fr in enumerate(fused, 1):
    t = fr.per_list_rank.get("text", "—")
    v = fr.per_list_rank.get("visual", "—")
    eid = fr.record.evidence_id[:28]
    print(f"{rank:<6}{eid:<30}{fr.source_modality:<10}{fr.rrf_score:<10.4f}{str(t):<8}{v}")

---
## 07b.7 Compare Retrieval Methods

We now compare four retrieval strategies on the same query:

| Strategy | Text | Caption | CLIP |
|---|:-:|:-:|:-:|
| Text-only | ✓ | | |
| Caption-based | ✓ | ✓ | |
| CLIP-only | | | ✓ |
| Fused (text + visual) | ✓ | | ✓ |

In [ ]:
def compare_retrieval(query: str, k: int = 5):
    print(f"Query: '{query}'")
    print("=" * 60)

    # 1. Text-only
    text_only = MultimodalRetriever(vector_store=text_store)
    r_text = text_only.retrieve(query, k_text=k, k_final=k)
    print(f"\n[Text-only] {len(r_text)} results")
    for r in r_text[:3]:
        print(f"  page={r.page}  mod={r.modality}  score={r.retrieval_score:.4f}")

    # 2. Fused text + visual (if visual index is populated)
    if visual_store.size > 0:
        fused = MultimodalRetriever(
            vector_store=text_store, visual_store=visual_store
        )
        r_fused = fused.retrieve(query, k_text=k, k_visual=k, k_final=k)
        n_text = sum(1 for r in r_fused if r.modality == "text")
        n_img = sum(1 for r in r_fused if r.modality == "image")
        print(f"\n[Text + CLIP] {len(r_fused)} results  ({n_text} text, {n_img} image)")
        for r in r_fused[:3]:
            fig = f"fig={r.figure_index}" if r.figure_index is not None else ""
            print(f"  page={r.page}  mod={r.modality}  {fig}  score={r.retrieval_score:.4f}")
    else:
        print("\n[Text + CLIP] — visual index empty")
    print()


compare_retrieval("What is the role of the attention mechanism?")
compare_retrieval("architecture diagram showing encoder and decoder")

---
## Summary

| What we built | Where it lives |
|---|---|
| CLIP image + text embeddings | `CLIPEmbedder` |
| FAISS index for visual evidence | `VisualVectorStore` |
| Rank-level fusion across incompatible score spaces | `reciprocal_rank_fusion` |
| Single retriever over all modalities | `MultimodalRetriever` |

**Key architectural insight:**
RRF fuses rankings, not raw scores — so text and visual evidence can be combined
even though they live in completely different embedding spaces with different score scales.

> We can now retrieve the correct visual evidence.
> The next step (Phase 07c) is to pass the original images **and** text to the VLM
> and produce a grounded answer that cites both forms of evidence.